# Experiment 4: ML Modeling & Experiment Tracking with MLflow
**Course**: Applied Data Science (ADS)  
**Dataset**: Twitter Customer Support (TWCS) Cleaned Dataset  
**Aim**: Build ML pipeline, tune hyperparameters, track experiments with MLflow.  

---
## Objectives
1. **Dataset Preparation**: Split dataset into 80% training and 20% testing sets using stratified sampling.
2. **Baseline Model Training**: Train 5 diverse machine learning algorithms spanning probabilistic, linear, max-margin, bagging, and gradient boosting paradigms (Multinomial Naive Bayes, Logistic Regression, Linear SVM, Random Forest, LightGBM).
3. **Hyperparameter Tuning**: Apply `GridSearchCV` on selected models to optimize decision thresholds and capacity.
4. **Experiment Tracking with MLflow**: Track runs, log hyperparameters, evaluation metrics (Accuracy, Macro F1, Weighted F1), and model artifacts.
5. **Model Selection & Saving**: Select the top-performing champion model and serialize it for production inference.

## 1. Imports & Environment Setup

In [ ]:
import os
import time
import json
from pathlib import Path

import numpy as np
import pandas as pd
import scipy.sparse as sp
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import StandardScaler, MinMaxScaler
from sklearn.naive_bayes import MultinomialNB
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, classification_report, confusion_matrix
import joblib

from lightgbm import LGBMClassifier
import mlflow
import mlflow.sklearn

from src.preprocessing import clean_tweet_text, apply_negation_tagging
from src.emotion_labeler import EmotionLabeler, EMOTION_CLASSES

# Configure MLflow Tracking
mlflow_dir = Path('../mlruns').resolve()
os.makedirs(mlflow_dir, exist_ok=True)
mlflow.set_tracking_uri(f"file:///{str(mlflow_dir).replace('\\', '/')}")
mlflow.set_experiment("Customer_Support_Emotion_Classification_Exp4")
print("MLflow Tracking configured successfully at:", mlflow.get_tracking_uri())

## 2. Dataset Preparation & Feature Engineering
We load the cleaned Twitter customer interactions and perform an 80/20 stratified split to preserve emotion class proportions.
Text features are extracted using **TF-IDF with unigrams and bigrams**, combined with dense **VADER sentiment polarity scores**.

In [ ]:
data_path = Path('../data/processed/twcs_cleaned.csv')
df = pd.read_csv(data_path)
print(f"Loaded {len(df):,} total interactions.")

# Sample 25,000 interactions for fast and representative training
df_sample = df.sample(n=25000, random_state=42).reset_index(drop=True)
labeler = EmotionLabeler()
df_labeled = labeler.label_dataframe(df_sample, text_column='clean_text')
df_labeled['negated_text'] = df_labeled['clean_text'].apply(apply_negation_tagging)

# 80/20 Stratified Split
train_df, test_df = train_test_split(
    df_labeled,
    test_size=0.20,
    random_state=42,
    stratify=df_labeled['emotion']
)
print(f"Train set: {len(train_df):,} samples | Test set: {len(test_df):,} samples")

# Vectorize with TF-IDF
vectorizer = TfidfVectorizer(ngram_range=(1, 2), min_df=2, max_features=10000, sublinear_tf=True)
X_tr_tfidf = vectorizer.fit_transform(train_df['negated_text'])
X_te_tfidf = vectorizer.transform(test_df['negated_text'])

# Scale numerical VADER sentiment features
vader_cols = ['vader_compound', 'vader_pos', 'vader_neg', 'vader_neu']
scaler = StandardScaler()
X_tr_vader = scaler.fit_transform(train_df[vader_cols].values)
X_te_vader = scaler.transform(test_df[vader_cols].values)

# Combine sparse lexical + dense polarity features
X_train = sp.hstack([X_tr_tfidf, X_tr_vader], format='csr')
X_test = sp.hstack([X_te_tfidf, X_te_vader], format='csr')
y_train = train_df['emotion'].values
y_test = test_df['emotion'].values
print("Combined Feature Matrix Shape:", X_train.shape)

## 3. Baseline Model Training & MLflow Tracking
We train 5 distinct baseline models and track all parameters, metrics, and models with MLflow:
1. **Multinomial Naive Bayes** (Fast probabilistic counting baseline)
2. **Logistic Regression** (Linear probability model)
3. **Linear SVM** (Maximum-margin hyperplane classifier)
4. **Random Forest** (Ensemble bagging of decision trees)
5. **LightGBM** (State-of-the-art gradient boosted trees)

In [ ]:
# Train and log baselines to MLflow
from src.experiment_4_modeling import train_and_track_experiments

# Run complete pipeline
summary = train_and_track_experiments()
print("Experiment 4 Pipeline Execution Complete!")

## 4. Benchmark Evaluation & Visual Diagnostics
Let us inspect the comparative performance across all architectures.

In [ ]:
benchmark_df = pd.DataFrame(summary['all_benchmarks'])
display(benchmark_df[['Model', 'Stage', 'Family', 'Accuracy', 'Macro F1', 'Weighted F1', 'Training Time (s)']])

### Visualizations Generated:
- **Model Performance Benchmark** (`plots/exp4_model_comparison.png`)
- **Confusion Matrix Diagnostics** (`plots/exp4_confusion_matrices.png`)
- **Hyperparameter Tuning Deltas** (`plots/exp4_tuning_comparison.png`)
- **MLflow Tracking Dashboard** (`plots/exp4_mlflow_dashboard.png`)

In [ ]:
from IPython.display import Image, display
display(Image(filename='../plots/exp4_model_comparison.png'))
display(Image(filename='../plots/exp4_confusion_matrices.png'))

## 5. Conclusion & Model Production Deployment
The champion model pipeline has been saved to `models/best_emotion_model_exp4.joblib` and logged in MLflow.
It bundles text preprocessing, negation tagging, TF-IDF vectorization, feature scaling, and inference in a single self-contained artifact.